In [1]:
# --- Safe paths for CI & local ---
import os
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Определим каталог reports независимо от того, где запущен ноутбук
CWD = Path.cwd()
REPORTS = (CWD.parent / "reports") if CWD.name == "notebooks" else (CWD / "reports")
REPORTS.mkdir(parents=True, exist_ok=True)

print("Working dir:", CWD)
print("Reports dir:", REPORTS)

Working dir: C:\Users\Natalia\PycharmProjects\trailflow\notebooks
Reports dir: C:\Users\Natalia\PycharmProjects\trailflow\reports


In [2]:
# --- Bootstrap: make the notebook CI-safe and reproducible ---
import os, sys, math, json, random
import numpy as np
import pandas as pd

# 1) Headless plotting in CI (no X server)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 2) Determinism
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# 3) Lightweight mode in CI: keep runs fast
IS_CI = os.getenv("CI", "").lower() == "true"
N = 3000 if IS_CI else 20000       # smaller sample size in CI
TIMEOUT_S = 60 if IS_CI else 300   # any sleeps/simulations must respect this

# 4) Optional local imports (roi helpers) with fallback
# Try to import your helper; if not present, define minimal ENB calc inline
try:
    sys.path.append("src")
    from trialflow.roi import enb_from_scores  # if you have it
except Exception:
    def enb_from_scores(y_true, p_pred, k_percent, cost_no_show=200.0, cost_intervention=1.5, uplift=0.30):
        """Minimal ENB: target top-K% by risk and compute expected net benefit."""
        n = len(y_true)
        m = max(1, int(n * (k_percent / 100.0)))
        order = np.argsort(-p_pred)
        p_top = np.asarray(p_pred)[order][:m]
        benefit = uplift * float(p_top.sum()) * cost_no_show
        cost = m * cost_intervention
        return benefit - cost

print({"IS_CI": IS_CI, "N": N})

{'IS_CI': False, 'N': 20000}


In [3]:
# --- Minimal synthetic data for ROI curve (no external files) ---
# Simulate probabilities with mild signal so that ENB curve is meaningful
base_risk = 0.12
x1 = np.random.normal(0, 1, N)
x2 = np.random.normal(0, 1, N)
logit = np.log(base_risk/(1-base_risk)) + 0.5*x1 + 0.2*x2 + np.random.normal(0, 0.5, N)
p = 1/(1+np.exp(-logit))
y = np.random.binomial(1, p)

# Build ROI curve for K in 5..50%
Ks = list(range(5, 55, 5))
enb = [enb_from_scores(y, p, k) for k in Ks]

plt.figure()
plt.plot(Ks, enb, marker="o")
plt.xlabel("Top-K% target budget")
plt.ylabel("Expected Net Benefit (€)")
plt.title("ROI curve (synthetic)")
plt.tight_layout()


out_path = REPORTS/"roi_curve_from_notebook.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight")
print("Saved figure to:", out_path)
#plt.savefig("reports/roi_curve_from_notebook.png", dpi=120)
enb[:3], len(enb)


Saved figure to: C:\Users\Natalia\PycharmProjects\trailflow\reports\roi_curve_from_notebook.png


([21713.176573477456, 37275.67082438582, 50272.08651249034], 10)